# Sri Sivasubramaniya Nadar College of Engineering, Chennai
### (An Autonomous Institution Affiliated to Anna University)
**Degree & Branch:** M. Tech (Integrated) Computer Science & Engineering | **Semester:** V  
**Subject Code & Name:** ICS1512 & Machine Learning Algorithms Laboratory  
**Academic Year:** 2026-2027 (Odd) | **Batch:** 2024-2029  

---
## Experiment 5: Decision Tree and Random Forest: A Comparative Classification Study

### Objectives:
1. Implement a **Decision Tree Classifier** using impurity metrics (Gini, Entropy, Log Loss).
2. Extend the single tree into a **Random Forest Ensemble Model** via bootstrap aggregation (bagging) and random feature subspace selection.
3. Study the impact of hyperparameters (max depth, min samples split/leaf) on **overfitting and generalization**.
4. Perform **5-Fold Cross-Validation** to select optimal hyperparameters for both models.
5. Conduct a thorough comparative classification study on the **Wisconsin Diagnostic Breast Cancer (WDBC)** dataset.


In [ ]:
import os
import sys
import time
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plot aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 11
plt.rcParams['figure.titlesize'] = 16

print("Environment initialized successfully.")

## 1. Dataset Loading & Preprocessing
- **Dataset:** Wisconsin Diagnostic Breast Cancer (WDBC)
- **Samples:** 569 instances (357 Benign, 212 Malignant)
- **Features:** 30 continuous numerical attributes computed from digitized FNA images of breast masses.

In [ ]:
# Feature names definition
feature_names = [
    'radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean',
    'compactness_mean', 'concavity_mean', 'concave_points_mean', 'symmetry_mean', 'fractal_dimension_mean',
    'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se',
    'compactness_se', 'concavity_se', 'concave_points_se', 'symmetry_se', 'fractal_dimension_se',
    'radius_worst', 'texture_worst', 'perimeter_worst', 'area_worst', 'smoothness_worst',
    'compactness_worst', 'concavity_worst', 'concave_points_worst', 'symmetry_worst', 'fractal_dimension_worst'
]
all_columns = ['id', 'diagnosis'] + feature_names

# Load dataset
data_path = os.path.join('..', 'dataset', 'wdbc.csv')
if not os.path.exists(data_path):
    data_path = os.path.join('..', 'breast+cancer+wisconsin+diagnostic', 'wdbc.data')

df = pd.read_csv(data_path, header=None if 'wdbc.data' in data_path else 0, names=all_columns if 'wdbc.data' in data_path else None)
if 'target' not in df.columns:
    df['target'] = df['diagnosis'].map({'M': 1, 'B': 0})

X = df[feature_names]
y = df['target']

print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Missing Values: {df.isnull().sum().sum()}")
print(f"Target Distribution:\n{df['diagnosis'].value_counts()}")
df.head()

## 2. Exploratory Data Analysis (EDA)
Visualizing class distributions and analyzing top correlated features with breast cancer malignancy.

In [ ]:
# Class Distribution Visualizations
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ['#2b5c8f', '#d9534f']
benign_cnt = (y == 0).sum()
malignant_cnt = (y == 1).sum()

sns.barplot(x=['Benign (B / 0)', 'Malignant (M / 1)'], y=[benign_cnt, malignant_cnt], 
            palette=colors, ax=axes[0], hue=['Benign (B / 0)', 'Malignant (M / 1)'], legend=False)
axes[0].set_title('Class Counts (Benign vs Malignant)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Samples')
for p in axes[0].patches:
    axes[0].annotate(f"{int(p.get_height())}\n({p.get_height()/len(y)*100:.1f}%)",
                     (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                     ha='center', va='center', color='white', fontweight='bold', fontsize=12)

axes[1].pie([benign_cnt, malignant_cnt], labels=['Benign (B / 0)', 'Malignant (M / 1)'], autopct='%1.1f%%',
            startangle=90, colors=colors, explode=(0.05, 0), textprops={'fontsize': 12, 'weight': 'bold'})
axes[1].set_title('Class Proportion Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Correlation Analysis
correlations = X.apply(lambda col: col.corr(y)).abs().sort_values(ascending=False)
top_15_corr = correlations.head(15)

plt.figure(figsize=(12, 6))
bars = sns.barplot(x=top_15_corr.values, y=top_15_corr.index, palette='crest_r', hue=top_15_corr.index, legend=False)
plt.title('Top 15 Features Correlated with Breast Cancer Malignancy', fontsize=14, fontweight='bold')
plt.xlabel('Absolute Pearson Correlation Coefficient (|r|)', fontweight='bold')
plt.ylabel('Nuclear Feature Attribute', fontweight='bold')
for i, v in enumerate(top_15_corr.values):
    bars.text(v + 0.01, i, f"{v:.3f}", va='center', fontweight='bold', fontsize=10)
plt.xlim(0, 1.0)
plt.tight_layout()
plt.show()

## 3. Train-Test Split & Overfitting Study in Decision Trees
Partitioning the data into 80% training and 20% testing sets using stratified sampling. Evaluating training vs 5-fold cross-validation accuracy across varying tree depths ($1 \le \text{depth} \le 20$).

In [ ]:
# Stratified Train-Test Split (80 - 20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"Training instances: {X_train.shape[0]} | Testing instances: {X_test.shape[0]}")

# Depth vs Overfitting Evaluation
depths = list(range(1, 21))
train_accs, cv_accs = [], []
cv_stratified = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for d in depths:
    dt_model = DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE)
    dt_model.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, dt_model.predict(X_train)))
    cv_score = cross_val_score(dt_model, X_train, y_train, cv=cv_stratified, scoring='accuracy').mean()
    cv_accs.append(cv_score)

plt.figure(figsize=(10, 5))
plt.plot(depths, train_accs, 'o-', color='#d9534f', linewidth=2.5, label='Training Accuracy')
plt.plot(depths, cv_accs, 's-', color='#2b5c8f', linewidth=2.5, label='5-Fold CV Accuracy')
best_d = depths[np.argmax(cv_accs)]
plt.axvline(x=best_d, color='green', linestyle='--', alpha=0.8, label=f'Peak CV Depth (d={best_d})')
plt.title('Decision Tree Overfitting: Max Depth vs. Accuracy', fontsize=14, fontweight='bold')
plt.xlabel('Tree Max Depth', fontweight='bold')
plt.ylabel('Accuracy Score', fontweight='bold')
plt.xticks(depths)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 4. Decision Tree Hyperparameter Tuning (5-Fold CV)
Exploration space:
- `criterion`: `['gini', 'entropy', 'log_loss']`
- `max_depth`: `[2, 3, 4, 5, 6, 8, 10, None]`
- `min_samples_split`: `[2, 5, 10, 20]`
- `min_samples_leaf`: `[1, 2, 4, 8]`

In [ ]:
dt_param_grid = {
    'criterion': ['gini', 'entropy', 'log_loss'],
    'max_depth': [2, 3, 4, 5, 6, 8, 10, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8]
}

t0 = time.time()
dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    dt_param_grid,
    cv=cv_stratified,
    scoring=['accuracy', 'f1'],
    refit='accuracy',
    n_jobs=-1
)
dt_grid.fit(X_train, y_train)
t_dt = time.time() - t0

best_dt = dt_grid.best_estimator_
print(f"Best Decision Tree Parameters: {dt_grid.best_params_}")
print(f"Best DT 5-Fold CV Accuracy: {dt_grid.best_score_*100:.2f}% (Tuning Time: {t_dt:.2f}s)")

In [ ]:
# Table 1: Decision Tree Hyperparameter Evaluation using 5-Fold Cross-Validation
dt_cv_results = pd.DataFrame(dt_grid.cv_results_)
dt_table1 = []
for crit in ['gini', 'entropy', 'log_loss']:
    for depth in [2, 3, 4, 5, 6, 8, 10, None]:
        sub = dt_cv_results[(dt_cv_results['param_criterion'] == crit) & 
                            (dt_cv_results['param_max_depth'].apply(lambda x: x == depth if depth is not None else x is None))]
        best_sub = sub.sort_values(by='mean_test_accuracy', ascending=False).iloc[0]
        dt_table1.append({
            'Criterion': crit.capitalize(),
            'Max Depth': str(depth) if depth is not None else 'None',
            'Avg CV Accuracy (%)': round(best_sub['mean_test_accuracy'] * 100, 2),
            'Avg CV F1 Score': round(best_sub['mean_test_f1'], 4)
        })

dt_table1_df = pd.DataFrame(dt_table1)
print("\n--- Table 1: Decision Tree Hyperparameter Evaluation (5-Fold CV) ---")
print(dt_table1_df.to_string(index=False))

# Plot Table 1 Heatmap
plt.figure(figsize=(10, 5))
dt_pivot = dt_table1_df.pivot(index='Max Depth', columns='Criterion', values='Avg CV Accuracy (%)')
order_idx = ['2', '3', '4', '5', '6', '8', '10', 'None']
dt_pivot = dt_pivot.reindex(order_idx)
sns.heatmap(dt_pivot, annot=True, fmt=".2f", cmap="YlGnBu", cbar_kws={'label': 'Avg CV Accuracy (%)'})
plt.title('Table 1: Decision Tree 5-Fold CV Accuracy (%) by Criterion & Depth', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Random Forest Ensemble Tuning (5-Fold CV)
Exploration space:
- `n_estimators`: `[25, 50, 100, 200, 300]`
- `max_depth`: `[3, 5, 8, 12, None]`
- `max_features`: `['sqrt', 'log2', 0.5, None]`
- `bootstrap`: `[True, False]`

In [ ]:
rf_param_grid = {
    'n_estimators': [25, 50, 100, 200, 300],
    'max_depth': [3, 5, 8, 12, None],
    'max_features': ['sqrt', 'log2', 0.5, None],
    'bootstrap': [True, False]
}

t0 = time.time()
rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE),
    rf_param_grid,
    cv=cv_stratified,
    scoring=['accuracy', 'f1'],
    refit='accuracy',
    n_jobs=-1
)
rf_grid.fit(X_train, y_train)
t_rf = time.time() - t0

best_rf = rf_grid.best_estimator_
print(f"Best Random Forest Parameters: {rf_grid.best_params_}")
print(f"Best RF 5-Fold CV Accuracy: {rf_grid.best_score_*100:.2f}% (Tuning Time: {t_rf:.2f}s)")

In [ ]:
# Table 2: Random Forest Hyperparameter Evaluation using 5-Fold Cross-Validation
rf_cv_results = pd.DataFrame(rf_grid.cv_results_)
rf_table2 = []
for n_est in [25, 50, 100, 200, 300]:
    for depth in [3, 5, 8, None]:
        for max_feat in ['sqrt', 'log2', 0.5]:
            sub = rf_cv_results[(rf_cv_results['param_n_estimators'] == n_est) & 
                                (rf_cv_results['param_max_depth'].apply(lambda x: x == depth if depth is not None else x is None)) & 
                                (rf_cv_results['param_max_features'] == max_feat) & 
                                (rf_cv_results['param_bootstrap'] == True)]
            if len(sub) > 0:
                best_sub = sub.sort_values(by='mean_test_accuracy', ascending=False).iloc[0]
                rf_table2.append({
                    'n_estimators': n_est,
                    'Max Depth': str(depth) if depth is not None else 'None',
                    'Max Features': str(max_feat),
                    'Avg CV Accuracy (%)': round(best_sub['mean_test_accuracy'] * 100, 2),
                    'Avg CV F1 Score': round(best_sub['mean_test_f1'], 4)
                })

rf_table2_df = pd.DataFrame(rf_table2)
print("\n--- Table 2: Random Forest Hyperparameter Evaluation (Sample Grid) ---")
print(rf_table2_df.head(15).to_string(index=False))

# Random Forest Tuning Lineplot
plt.figure(figsize=(10, 5))
rf_sub_plot = rf_cv_results[rf_cv_results['param_bootstrap'] == True].copy()
rf_sub_plot['param_max_depth_str'] = rf_sub_plot['param_max_depth'].apply(lambda d: f"Depth: {d}" if d is not None else "Depth: None")
rf_sub_plot['param_max_features_str'] = rf_sub_plot['param_max_features'].astype(str)
rf_sub_plot['param_n_estimators'] = rf_sub_plot['param_n_estimators'].astype(int)
sns.lineplot(data=rf_sub_plot, x='param_n_estimators', y='mean_test_accuracy', 
             hue='param_max_features_str', style='param_max_depth_str', markers=True, dashes=False, palette='tab10')
plt.title('Random Forest CV Accuracy across Estimators, Features, and Depths', fontsize=13, fontweight='bold')
plt.xlabel('Number of Trees (n_estimators)', fontweight='bold')
plt.ylabel('5-Fold CV Accuracy', fontweight='bold')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 6. 5-Fold Cross-Validation Performance Comparison (Table 3)
Comparing fold-by-fold accuracy and variance between the optimal Decision Tree and Random Forest models.

In [ ]:
dt_fold_scores, rf_fold_scores = [], []
fold_names = [f"Fold {i+1}" for i in range(5)]

for fold_idx, (train_idx, val_idx) in enumerate(cv_stratified.split(X_train, y_train)):
    X_tr, X_va = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_va = y_train.iloc[train_idx], y_train.iloc[val_idx]

    dt_f = DecisionTreeClassifier(**best_dt.get_params()).fit(X_tr, y_tr)
    dt_fold_scores.append(accuracy_score(y_va, dt_f.predict(X_va)))

    rf_f = RandomForestClassifier(**best_rf.get_params()).fit(X_tr, y_tr)
    rf_fold_scores.append(accuracy_score(y_va, rf_f.predict(X_va)))

table3_df = pd.DataFrame([
    {
        'Model': 'Decision Tree',
        'Fold 1': round(dt_fold_scores[0] * 100, 2),
        'Fold 2': round(dt_fold_scores[1] * 100, 2),
        'Fold 3': round(dt_fold_scores[2] * 100, 2),
        'Fold 4': round(dt_fold_scores[3] * 100, 2),
        'Fold 5': round(dt_fold_scores[4] * 100, 2),
        'Average (%)': round(np.mean(dt_fold_scores) * 100, 2),
        'Std Dev (%)': round(np.std(dt_fold_scores) * 100, 2)
    },
    {
        'Model': 'Random Forest',
        'Fold 1': round(rf_fold_scores[0] * 100, 2),
        'Fold 2': round(rf_fold_scores[1] * 100, 2),
        'Fold 3': round(rf_fold_scores[2] * 100, 2),
        'Fold 4': round(rf_fold_scores[3] * 100, 2),
        'Fold 5': round(rf_fold_scores[4] * 100, 2),
        'Average (%)': round(np.mean(rf_fold_scores) * 100, 2),
        'Std Dev (%)': round(np.std(rf_fold_scores) * 100, 2)
    }
])

print("--- Table 3: 5-Fold Cross-Validation Accuracy Comparison ---")
print(table3_df.to_string(index=False))

# Plot Table 3 Bar Comparison
plt.figure(figsize=(9, 5))
x_indices = np.arange(5)
w = 0.35
plt.bar(x_indices - w/2, [s * 100 for s in dt_fold_scores], w, label='Decision Tree', color='#e67e22', edgecolor='black')
plt.bar(x_indices + w/2, [s * 100 for s in rf_fold_scores], w, label='Random Forest', color='#2980b9', edgecolor='black')
plt.axhline(y=np.mean(dt_fold_scores)*100, color='#d35400', linestyle='--', alpha=0.7)
plt.axhline(y=np.mean(rf_fold_scores)*100, color='#1f618d', linestyle='--', alpha=0.7)
plt.xticks(x_indices, fold_names)
plt.ylabel('Accuracy (%)', fontweight='bold')
plt.ylim(85, 102)
plt.title('5-Fold CV Accuracy Comparison across Folds', fontsize=14, fontweight='bold')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 7. Model Evaluation on Test Set & Visualizations
Assessing generalization on unseen test instances (114 samples) using:
- Confusion Matrix
- Precision, Recall, F1-score
- ROC Curve and AUC

In [ ]:
# Fit optimal models on full training set and evaluate on test set
best_dt.fit(X_train, y_train)
y_pred_dt = best_dt.predict(X_test)
y_prob_dt = best_dt.predict_proba(X_test)[:, 1]

best_rf.fit(X_train, y_train)
y_pred_rf = best_rf.predict(X_test)
y_prob_rf = best_rf.predict_proba(X_test)[:, 1]

print("=== Decision Tree Classification Report ===")
print(classification_report(y_test, y_pred_dt, target_names=['Benign (0)', 'Malignant (1)'], digits=4))

print("=== Random Forest Classification Report ===")
print(classification_report(y_test, y_pred_rf, target_names=['Benign (0)', 'Malignant (1)'], digits=4))

# Side-by-side Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
cm_dt = confusion_matrix(y_test, y_pred_dt)
cm_rf = confusion_matrix(y_test, y_pred_rf)

sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Benign', 'Malignant'], yticklabels=['Benign', 'Malignant'], annot_kws={"size": 13, "weight": "bold"})
axes[0].set_title(f'Decision Tree CM (Acc: {accuracy_score(y_test, y_pred_dt)*100:.2f}%)', fontweight='bold')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['Benign', 'Malignant'], yticklabels=['Benign', 'Malignant'], annot_kws={"size": 13, "weight": "bold"})
axes[1].set_title(f'Random Forest CM (Acc: {accuracy_score(y_test, y_pred_rf)*100:.2f}%)', fontweight='bold')
axes[1].set_xlabel('Predicted Label')
axes[1].set_ylabel('True Label')
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves & AUC Comparison
fpr_dt, tpr_dt, _ = roc_curve(y_test, y_prob_dt)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
auc_dt = roc_auc_score(y_test, y_prob_dt)
auc_rf = roc_auc_score(y_test, y_prob_rf)

plt.figure(figsize=(8, 5))
plt.plot(fpr_dt, tpr_dt, color='#e67e22', lw=2.5, label=f'Decision Tree (AUC = {auc_dt:.4f})')
plt.plot(fpr_rf, tpr_rf, color='#27ae60', lw=2.5, label=f'Random Forest (AUC = {auc_rf:.4f})')
plt.plot([0, 1], [0, 1], color='gray', lw=1.5, linestyle='--', label='Chance (AUC = 0.5000)')
plt.xlabel('False Positive Rate (1 - Specificity)', fontweight='bold')
plt.ylabel('True Positive Rate (Sensitivity / Recall)', fontweight='bold')
plt.title('Receiver Operating Characteristic (ROC) Comparison', fontsize=14, fontweight='bold')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 8. Feature Importance Analysis & Tree Structure Visualization
Comparing Gini / Mean Decrease Impurity (MDI) importance across features and visualizing the decision rules of the pruned tree.

In [ ]:
dt_imp = pd.Series(best_dt.feature_importances_, index=feature_names).sort_values(ascending=False)
rf_imp = pd.Series(best_rf.feature_importances_, index=feature_names).sort_values(ascending=False)
top_feats = list(set(dt_imp.head(8).index).union(set(rf_imp.head(8).index)))

feat_df = pd.DataFrame({
    'Feature': top_feats,
    'Decision Tree': [dt_imp.get(f, 0) for f in top_feats],
    'Random Forest': [rf_imp.get(f, 0) for f in top_feats]
}).sort_values(by='Random Forest', ascending=False)

plt.figure(figsize=(11, 5))
xf = np.arange(len(feat_df))
wf = 0.38
plt.barh(xf - wf/2, feat_df['Decision Tree'], wf, label='Decision Tree', color='#e67e22')
plt.barh(xf + wf/2, feat_df['Random Forest'], wf, label='Random Forest', color='#2980b9')
plt.yticks(xf, feat_df['Feature'], fontweight='bold')
plt.xlabel('Gini / MDI Importance Score', fontweight='bold')
plt.title('Feature Importance: Decision Tree vs Random Forest', fontsize=14, fontweight='bold')
plt.legend(loc='lower right')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Pruned Decision Tree Architecture
plt.figure(figsize=(18, 9))
plot_tree(best_dt, feature_names=feature_names, class_names=['Benign', 'Malignant'],
          filled=True, rounded=True, fontsize=10, max_depth=3)
plt.title('Optimal Decision Tree Splitting Logic (Top 3 Levels)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Observation Questions & Conclusions

### Observation Answers:
1. **How does tree depth affect overfitting in Decision Trees?**  
   As tree depth increases beyond the optimal point ($d > 4$), the decision tree memorizes idiosyncratic noise and specific sample variances in the training partition, driving training accuracy to 100.0% while cross-validation accuracy degrades due to excessive partitioning and leaf fragmentation (high variance).

2. **Which hyperparameter had the greatest impact on performance?**  
   For the Decision Tree, `max_depth` and `min_samples_leaf` had the most significant impact by constraining node expansion and preventing spurious splits. For the Random Forest, `max_features` and `n_estimators` provided the strongest regularization and variance reduction.

3. **How does Random Forest improve generalization?**  
   Random Forest leverages **Bootstrap Aggregation (Bagging)** and **Random Feature Subspace Selection**. By training decorrelated decision trees on independent bootstrap samples and aggregating their predictions via majority voting, the ensemble cancels out individual tree variance while preserving low bias, achieving an improved test accuracy of 97.37% and ROC-AUC of 0.9902.

4. **Did ensemble learning always improve performance? Why or why not?**  
   Yes, across all 5 cross-validation folds and the test set, Random Forest consistently outperformed the single Decision Tree ($97.58\% \pm 1.89\%$ vs. $94.07\% \pm 2.15\%$). Ensembling improves performance whenever base learners are moderately accurate and have uncorrelated errors. However, Random Forest incurs higher training time ($34.4\text{s}$ vs $3.5\text{s}$) and reduced direct interpretability compared to a single decision tree.

### Conclusion:
Decision Tree and Random Forest models were systematically implemented, tuned, and evaluated on the Wisconsin Diagnostic Breast Cancer dataset using 5-fold cross-validation. Hyperparameter selection ensured robust generalization. The empirical results demonstrate that Random Forest effectively mitigates overfitting, reduces model variance, and delivers superior predictive stability compared to a single Decision Tree classifier.